## 📌 Image Editing Pipeline (SDXL, Colab)

В данном ноутбуке реализован pipeline генерации датасета изображений
с использованием модели diffusers/sdxl-instructpix2pix-768.

Запуск выполняется на GPU (Google Colab), так как локальное оборудование
не позволяет стабильно использовать данную модель.

In [ ]:
!pip install -q diffusers transformers accelerate safetensors datasets pillow tqdm

In [ ]:
!nvidia-smi

## 🧠 Загрузка модели

Используется модель SDXL InstructPix2Pix для редактирования изображений по текстовой инструкции.

Особенности:
- модель требует GPU (~12GB)
- используется float16 для экономии памяти
- включены оптимизации (attention slicing, VAE slicing)

In [ ]:
import torch
from diffusers import StableDiffusionXLInstructPix2PixPipeline
from PIL import Image
import requests
from io import BytesIO

print("Loading model...")

pipe = StableDiffusionXLInstructPix2PixPipeline.from_pretrained(
    "diffusers/sdxl-instructpix2pix-768",
    torch_dtype=torch.float16,
    use_safetensors=True,
).to("cuda")

pipe.enable_attention_slicing()
pipe.vae.enable_slicing()

print("Model loaded")

In [ ]:
from PIL import Image
import requests
from io import BytesIO

url = "https://images.unsplash.com/photo-1501004318641-b39e6451bec6"

response = requests.get(url)
image = Image.open(BytesIO(response.content)).convert("RGB")
image = image.resize((768, 768))

image

In [ ]:
prompt = "make the image brighter and more colorful"

generator = torch.Generator(device="cuda").manual_seed(42)

result = pipe(
    prompt=prompt,
    image=image,
    height=768,
    width=768,
    guidance_scale=3.0,
    image_guidance_scale=1.5,
    num_inference_steps=30,
    generator=generator,
)

result.images[0]

## ⚙️ Генерация датасета

Шаги:
1. Берём изображения из датасета beans
2. Случайным образом выбираем 20 изображений
3. Генерируем инструкцию редактирования
4. Прогоняем через модель
5. Сохраняем:
   - исходные изображения
   - отредактированные изображения
   - JSON с метаданными

In [ ]:
import json
import random
import shutil
from pathlib import Path

import torch
from datasets import load_dataset
from tqdm import tqdm
from google.colab import files


DATASET_NAME = "beans"
SPLIT = "train"
LIMIT = 20
SEED = 42

RAW_DIR = Path("data/raw")
EDITED_DIR = Path("data/edited")
METADATA_DIR = Path("data/metadata")
OUTPUT_JSON_PATH = METADATA_DIR / "dataset.json"

EDIT_PROMPTS = [
    "Add small water droplets on the leaf.",
    "Make the image brighter and more colorful.",
    "Change the background to a sunny garden.",
    "Make the photo look like it was taken during golden hour.",
    "Improve the colors and contrast.",
]


RAW_DIR.mkdir(parents=True, exist_ok=True)
EDITED_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)

print("Loading dataset...")
dataset = load_dataset(DATASET_NAME, split=SPLIT)

indices = list(range(len(dataset)))
random.shuffle(indices)
selected_indices = indices[:LIMIT]

records = []

print("Processing images...")

'''Обработка изображений

Каждое изображение:
- приводится к размеру 768x768 (требование модели)
- получает случайный prompt
- обрабатывается моделью'''

for number, dataset_index in enumerate(tqdm(selected_indices), start=1):
    source_image = dataset[dataset_index]["image"].convert("RGB").resize((768, 768))

    source_image_id = f"source_{number:04d}"
    edited_image_id = f"edited_{number:04d}"

    source_image_path = RAW_DIR / f"{source_image_id}.jpg"
    edited_image_path = EDITED_DIR / f"{edited_image_id}.jpg"

    edit_prompt = random.choice(EDIT_PROMPTS)

    source_image.save(source_image_path)

    generator = torch.Generator(device="cuda").manual_seed(SEED + number)

    result = pipe(
        prompt=edit_prompt,
        image=source_image,
        height=768,
        width=768,
        guidance_scale=3.0,
        image_guidance_scale=1.5,
        num_inference_steps=30,
        generator=generator,
    )

    result.images[0].save(edited_image_path)

    records.append(
        {
            "source_image_id": source_image_id,
            "edit_prompt": edit_prompt,
            "edited_image_id": edited_image_id,
        }
    )

'''Формирование JSON

Для каждого изображения сохраняется:

- source_image_id
- edit_prompt
- edited_image_id

Формат соответствует требованиям задания.'''

with open(OUTPUT_JSON_PATH, "w", encoding="utf-8") as file:
    json.dump(records, file, ensure_ascii=False, indent=2)

print("Done")
print(f"Saved metadata to: {OUTPUT_JSON_PATH}")

shutil.make_archive("image_editing_dataset", "zip", "data")
files.download("image_editing_dataset.zip")

## ✅ Результат

Сформирован датасет:
- 20 исходных изображений
- 20 отредактированных изображений
- JSON с метаданными

Pipeline полностью воспроизводим.

## ⚠️ Замечание

Локально модель SDXL не использовалась из-за ограничений GPU (4GB VRAM).
В Colab (Tesla T4) модель работает стабильно.